# Step 1 — Install:



In [37]:
!pip install python-a2a


# Step 2 — Imports:



In [38]:
import threading
import time
from python_a2a import A2AServer, AgentCard, Message, TextContent, MessageRole, run_server, A2AClient
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


threading — lets you run servers in the background while the notebook keeps running.

time — used to pause so servers have time to start.

A2AServer — the base class for building an agent.

AgentCard — describes the agent (name, url, version). This is how other agents discover it.

Message, TextContent, MessageRole — the A2A message format.

run_server — starts the agent as an HTTP server.

A2AClient — the client that sends tasks to agents.

# Step 3 — Load the LLM:



In [39]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/flan-t5-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-large/0613663d0d48ea86ba8cb3d7a44f0f65dc596a2a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/flan-t5-large/0613663d0d48ea86ba8cb3d7a44f0f65dc596a2a/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/flan-t5-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-large/0613663d0d48ea86ba8cb3d7a44f0f65dc596a2a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/flan-t5-large/0613663d0d48ea86ba8cb3d7a44f0f65dc596a2a/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-large/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/flan-t5-large/resolve/main/spiece.model "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-large/xet-read-token/0613663d0d48ea86ba8cb3d7a44f0f65dc596a2a "HTTP/1.1 200 OK"


...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.13GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/flan-t5-large/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-large/0613663d0d48ea86ba8cb3d7a44f0f65dc596a2a/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/flan-t5-large/0613663d0d48ea86ba8cb3d7a44f0f65dc596a2a/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

This cell loads a small language model called flan-t5-base from Hugging Face. It does three things. First, it loads a tokenizer that converts text into tokens. Second, it loads the model itself, which is a sequence-to-sequence model designed for text-to-text tasks. Third, it defines a helper function called generate_text that takes a prompt, tokenizes it, runs the model, and decodes the output back into readable text. We use this helper inside the agents so they can produce real answers instead of hardcoded strings.

# Step 4 — Define the Researcher Agent (with LLM):




In [40]:
class ResearcherAgent(A2AServer):
    def __init__(self):
        agent_card = AgentCard(
            name="ResearcherAgent",
            description="Searches for information about a topic.",
            url="http://localhost:5001",
            version="1.0.0"
        )
        super().__init__(agent_card=agent_card)

    def handle_message(self, message):
        topic = message.content.text
        prompt = f"Give three short facts about {topic}."
        findings = generate_text(prompt)
        return Message(
            content=TextContent(text=findings),
            role=MessageRole.AGENT
        )

class ResearcherAgent(A2AServer): — define agent, inherit from A2AServer

def __init__(self): — runs when agent is created

agent_card = AgentCard( — start building the agent's ID card

name=... — agent name

description=... — what the agent does

url=... — where the agent listens

version=... — version number

) — close the card

super().__init__(...) — pass the card to the base server

def handle_message(self, message): — runs when a message arrives

topic = message.content.text — get text from the message

prompt = f"..." — build a prompt with the topic

findings = generate_text(prompt) — call the LLM

return Message( — start the reply

content=TextContent(text=findings) — wrap the LLM output

role=MessageRole.AGENT — mark it as from an agent

) — return the reply

# Step 5 — Define the Writer Agent (with LLM):



In [41]:
class WriterAgent(A2AServer):
    def __init__(self):
        agent_card = AgentCard(
            name="WriterAgent",
            description="Takes research and writes a short paragraph.",
            url="http://localhost:5002",
            version="1.0.0"
        )
        super().__init__(agent_card=agent_card)

    def handle_message(self, message):
        research = message.content.text
        prompt = f"Write a short report based on this: {research}"
        article = generate_text(prompt)
        return Message(
            content=TextContent(text=article),
            role=MessageRole.AGENT
        )

Same pattern — the Writer calls the LLM to turn research into a report.



# Step 6 — Start the Researcher:



In [42]:
researcher = ResearcherAgent()
researcher_thread = threading.Thread(
    target=lambda: run_server(researcher, port=5001),
    daemon=True
)
researcher_thread.start()
print("Researcher agent running on port 5001")

Researcher agent running on port 5001
Starting A2A server on http://0.0.0.0:5001/a2a
Google A2A compatibility: Enabled
 * Serving Flask app 'python_a2a.server.http'
 * Debug mode: off


researcher = ResearcherAgent() — create the agent instance

researcher_thread = threading.Thread( — start creating a background thread

target=lambda: run_server(researcher, port=5001), — thread runs the server on port 5001

daemon=True — thread stops when the notebook closes

) — close the thread definition

researcher_thread.start() — start the thread (server goes live)

print("Researcher agent running on port 5001") — confirm it started

# Step 7 — Start the Writer:





In [43]:
writer = WriterAgent()
writer_thread = threading.Thread(
    target=lambda: run_server(writer, port=5002),
    daemon=True
)
writer_thread.start()
print("Writer agent running on port 5002")

time.sleep(3)  # give both servers time to start

Writer agent running on port 5002
Starting A2A server on http://0.0.0.0:5002/a2a
Google A2A compatibility: Enabled
 * Serving Flask app 'python_a2a.server.http'
 * Debug mode: off


Address already in use
Port 5001 is in use by another program. Either identify and stop that program, or start the server with a different port.
Address already in use
Port 5002 is in use by another program. Either identify and stop that program, or start the server with a different port.


writer = WriterAgent() — create the Writer agent instance

writer_thread = threading.Thread( — start creating a background thread

target=lambda: run_server(writer, port=5002), — thread runs the server on port 5002

daemon=True — thread stops when notebook closes

) — close thread definition

writer_thread.start() — start the thread (server goes live)

print("Writer agent running on port 5002") — confirm it started

time.sleep(3) — pause 3 seconds so both servers are ready


# Step 8 — Client sends a task to the Researcher:





In [44]:
topic = "the water cycle"
researcher_client = A2AClient("http://localhost:5001")
research_result = researcher_client.ask(topic)
print(f"[Researcher] returned: {research_result}\n")

INFO:werkzeug:127.0.0.1 - - [08/Oct/2026 18:31:28] "GET /.well-known/agent.json HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Oct/2026 18:31:32] "POST /tasks/send HTTP/1.1" 200 -


[Researcher] returned: the water cycle is a cycle in which water flows from the surface to the ocean floor



topic = "A2A Multi-Agent Systems" — set the task text to send to the agent

researcher_client = A2AClient("http://localhost:5001") — create a client connected to the Researcher agent

research_result = researcher_client.ask(topic) — send the topic to the agent and get the reply (a string)

print(f"[Researcher] returned: {research_result}\n") — print the reply with a label and a blank line

# Step 9 — Client sends the research to the Writer:





In [45]:
writer_client = A2AClient("http://localhost:5002")
final_result = writer_client.ask(research_result)
print(f"[Writer] returned: {final_result}")

INFO:werkzeug:127.0.0.1 - - [08/Oct/2026 18:31:32] "GET /.well-known/agent.json HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Oct/2026 18:31:35] "POST /tasks/send HTTP/1.1" 200 -


[Writer] returned: the water cycle is a cycle in which water flows from the surface to the ocean floor .


writer_client = A2AClient("http://localhost:5002") — connect a client to the Writer agent

final_result = writer_client.ask(research_result) — send the Researcher's output to the Writer, get the reply

print(f"[Writer] returned: {final_result}") — print the Writer's reply with a label